In [1]:
import pandas as pd
import numpy as np

import sys
sys.path.insert(1, '../../scripts/')
from human_me.utils.load_environmental_variables import *
prebuild = '/data2/hratch/human_me/prebuild/'

# Protein turnover

In [2]:
def explode(df_, explode_cols, sep = None, fill_value=float('nan'), preserve_index=False):
    '''Split entries with multiple values separated by a separator into multiple rows (or entries in lists)
    
    https://stackoverflow.com/questions/12680754/split-explode-pandas-dataframe-string-entry-to-separate-rows
    
    df_: pd.DataFrame
    explode_cols: list
        columns to split
    sep: separator
        if None, already in list format
    fill_value: 
        if empty entry, what to fill it with
    preserve_index: bool
        keep original index values for each new row
    
    '''
    res = None
    for col in explode_cols:
        if res is None:
            df = df_.copy()
        else:
            df = res.copy()
        
        df[col]=df[col].str.split(sep)

        idx_cols = df.columns.difference([col])
        # calculate lengths of lists
        lens = df[col].apply(lambda x: len(x))

        idx = np.repeat(df.index.values, lens)
        # create "exploded" DF
        res = (pd.DataFrame({
                    col_:np.repeat(df[col_].values, lens)
                    for col_ in idx_cols},
                    index=idx)
                 .assign(**{col_:np.concatenate(df.loc[lens>0, col_].values)
                                for col_ in [col]}))
        # append those rows that have empty lists
        if (lens == 0).any():
            # at least one list in cells is empty
            res = (res.append(df.loc[lens==0, idx_cols], sort=False)
                      .fillna(fill_value))
        # revert the original index order
        res = res.sort_index()
        # reset index if requested
        if not preserve_index:        
            res.reset_index(drop=True, inplace = True)
    return res

def expand_id(df_):
    # expand uniprot id
    df = df_.copy()
    df = df[df.Uniprot.notna()]
    df['Uniprot'] = df['Uniprot'].astype(str)
    df=explode(df[df['Uniprot'].notna()], explode_cols = ['Uniprot'], sep = ';')
    df=explode(df, explode_cols = ['Uniprot'], sep = '/')
    return df

def merge_redundant(df_):
    df = df_.groupby(by=df_.Uniprot, axis=0).median()
    if 'nan' in df.index.tolist():
        df.drop(index = ['nan'], inplace = True)
    return df

In [3]:
hela = pd.read_excel(prebuild + 'Cambridge_protein_turnover.xls', sheet_name = 0)
c2c12 = pd.read_excel(prebuild + 'Cambridge_protein_turnover.xls', sheet_name = 1)
mval = np.median(hela.kdeg.tolist() + c2c12.kdeg.tolist())
print('The median degradation rate before mapping IDs: {:.4f} (hrs)'.format(mval))

dfs = [hela, c2c12]

# only keep these columns
cols = ['half-life t1/2 in h', 'kdeg', 'Uniprot']
dfs = [df[cols] for df in dfs]

# expand uniprot id
dfs = [expand_id(df) for df in dfs]

# merge duplicate ids by median value
dfs = [merge_redundant(df) for df in dfs]

# combine
hela, c2c12 = dfs[0], dfs[1]
hela['cell_line'] = 'HeLA'
c2c12['cell_line'] = 'C2C12'
ap = pd.concat([hela, c2c12], axis = 0, ignore_index = False)
ap['UNIPROT_ID'] = ap.index
ap.reset_index(inplace = True, drop = True)

The median degradation rate before mapping IDs: 0.0183 (hrs)


#### Map to HGNC IDs

In [4]:
# conserve existing mapping (consider many to one)
psim_me = pd.read_hdf('/data2/hratch/human_me/temp_psim.h5', key = 'n_exons') # ok to use final psim also
temp = psim_me[psim_me.UNIPROT_ID.isin(ap.UNIPROT_ID)][['UNIPROT_ID', 'HGNC_ID']]

mapper = dict()
for i in temp.index:
    hgnc_id = temp.loc[i, 'HGNC_ID']
    uniprot_id = temp.loc[i, 'UNIPROT_ID']
    
    if uniprot_id in mapper:
        mapper[uniprot_id] += [hgnc_id]
    else:
        mapper[uniprot_id] = [hgnc_id]

mapper = {k: ';'.join(sorted(set(v))) for k,v in mapper.items()}
ap['HGNC_ID'] = ap.UNIPROT_ID.map(mapper)

ap['HGNC_ID'] = ap.HGNC_ID.astype(str)
ap = explode(ap, explode_cols = ['HGNC_ID'], sep = ';', fill_value=float('nan'), preserve_index=False)
ap.replace('nan', float('nan'), inplace = True)

In [13]:
p1 = ap[ap.HGNC_ID.notna()]
p2 = ap[ap.HGNC_ID.isna()]

mapper = pd.read_csv(prebuild + 'Cambridge_uniprot_to_hgnc.txt', sep = '\t')
p2['HGNC_ID'] = p2.UNIPROT_ID.map(dict(zip(mapper.From, mapper.To)))
ap = pd.concat([p1, p2], axis = 0, ignore_index = True)
ap.to_csv(build_files_path + 'protein_turnover.csv')